# Mercado: Lieferqualität und Kundenservice
 **DABI1 · Projektaufgabe**

Gruppe | Studierende 1 | Studierende 2 | Studierende 3 |
------ | ------ | ------ |-----|
`8` | `Damian Frehar` | `Nils Hahn`|`Maximilian Hutter` |

## 2. Ihr Beratungsauftrag

### 2.1 Daten vorbereiten

#### 2.1 a Datenqualität prüfen

##### Import CSV-Dateien und Python-Bibliotheken

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import pprint as pp
import missingno as msno

In [ ]:
# Dictionary für die DataFrames
dataframes = {}

data_folder = Path("data")

# Alle CSV-Dateien im Ordner durchlaufen
for file in data_folder.glob("*.csv"):
    df = pd.read_csv(file, dtype={"customer_zip_code_prefix": str, "seller_zip_code_prefix": str, "geolocation_zip_code_prefix": str})
    dataframes[file.stem] = df
    print(file)

In [ ]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

##### Orders.csv

In [ ]:
orders = dataframes["orders"]
orders

In [ ]:
orders.dtypes

In [ ]:
orders.head()

###### Datentypen
(Konvertierung in Datatime)

In [ ]:
orders = dataframes["orders"]

datumsspalten = ["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"]

for spalte in datumsspalten:
    orders[spalte] = pd.to_datetime(orders[spalte], format="%Y-%m-%d %H:%M:%S")

orders.dtypes

###### Schlüssel

In [ ]:
schluessel = [
        col for col in orders.columns
        if orders[col].notna().all()
        and orders[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für orders ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für orders ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(orders)

Bei stornierten Bestellungen (canceled) fehlt das Lieferdatum beim Kunden fast immer, meist auch die Übergabe an den Transporteur. Das ist plausibel, weil die Ware nicht verschickt wurde. Ausnahme: 6 stornierte Bestellungen haben trotzdem ein Lieferdatum. Sie wurden vermutlich erst nach der Zustellung storniert, z. B. durch eine Rücksendung.

In [ ]:
orders[orders["order_status"] == "canceled"]

orders["not_approved"] = orders["order_approved_at"].isna()
px.histogram(orders, y="order_status", color="not_approved", width=1000)

In [ ]:
orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

In [ ]:
(orders["order_status"].eq("canceled") & orders["order_delivered_customer_date"].notna()).sum()

160 Bestellungen haben kein Freigabedatum (order_approved_at). Davon sind 141 canceled und 5 created; das ist plausibel, weil sie nie freigegeben wurden. Auffällig sind 14 Bestellungen mit Status delivered ohne Freigabedatum: Sie wurden geliefert, die Freigabe wurde aber nicht erfasst. Wir behalten sie, weil wir das Freigabedatum für die Merkmale (Lieferdauer, Verspätung) nicht brauchen.

###### Plausible Größenordnungen

In [ ]:
for spalte in datumsspalten:
    print(spalte)
    print(orders[spalte].min())
    print(orders[spalte].max())
    print("")

Kategorien von order status

In [ ]:
orders["order_status"].unique()

Fazit:
Datumspalten wurden in Datetime umgewandelt,
Fehlende Werte bei orders werden beibehalten, da die Quantität der fehlenden Daten überschauber ist, und die Werte aus einem plausiblen Grund fehlen,


##### Customers.csv

In [ ]:
customers = dataframes["customers"]

###### Datentypen

In [ ]:
customers.dtypes

In [ ]:
customers.head()

In [ ]:
customers["customer_city"].unique()

In [ ]:
customers["customer_state"].unique()

###### Schlüssel

In [ ]:
schluessel = [
        col for col in customers.columns
        if customers[col].notna().all()
        and customers[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für customers ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für customers ist die Spalte: {schluessel}")

macht sinn weil customer_unique_id ist mehrmals vorhanden da Kunden mehrere Bestellungen machen können

###### Fehlende Werte

In [ ]:
msno.matrix(customers)

keine fehlenden Werte :)

In [ ]:
a = 123
print("a")
print("Sometimes I dream of saving the world")

###### Plausible Größenordnungen

In [ ]:
px.histogram(customers, x="customer_state", color="customer_city")

sao paulo macht sinn weil das die größte Stadt ist

##### Items.csv

###### Datentypen

In [ ]:
items = dataframes["items"]

In [ ]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S")

In [ ]:
items.dtypes

In [ ]:
items

###### Schlüssel

In [ ]:
schluessel = [
        col for col in items.columns
        if items[col].notna().all()
        and items[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für items ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für items ist die Spalte: {schluessel}")

In [ ]:
items["order_id"].nunique()

In [ ]:
items["order_key"] = items["order_id"].astype(str) + "_" + items["order_item_id"].astype(str)

In [ ]:
items["order_key"].nunique()

In [ ]:
items["order_key"].duplicated(keep=False)

In [ ]:
items["order_key"].duplicated().sum()

In [ ]:
items[items["order_key"].duplicated(keep=False)].sort_values("order_key")

In [ ]:
items["order_key"].value_counts()[lambda x: x > 1]

In [ ]:
items.duplicated().sum()

Wir haben 225 Zeilen welche identische Duplikate sind, raus mit denen!

In [ ]:
items = items.drop_duplicates()

In [ ]:
items.duplicated().sum()

In [ ]:
schluessel = [
        col for col in items.columns
        if items[col].notna().all()
        and items[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für items ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für items ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(items)

keine fehlenden Werte in items.csv

###### Plausible Größenordnungen

Klären wir erstmal das offensichtlichste Vorab. Warum trag ich nh Brille?

Keine Ahnung...

Ist auch völlig uninteressant

In [ ]:
px.histogram(items, x="price")

In [ ]:
items["price"].describe()

In [ ]:
#Untersuchen was das teuerste Produkt ist
items.sort_values(by="price", ascending=False)

In [ ]:
products = dataframes["products"]
products[products["product_id"] == "489ae2aa008f021502940f251d4cce7f"]

joa wahrscheinlich passt des schon

In [ ]:
px.histogram(items, x="freight_value")

In [ ]:
items['freight_value'].describe()

In [ ]:
items["shipping_limit_date"].max()
items["shipping_limit_date"].min()

In [ ]:
items_merge = pd.merge(items, products, on="product_id")

In [ ]:

px.scatter(items_merge, x="product_weight_g", y="freight_value", title="Zusammenhang zwischen Gewicht und Frachtwert")

##### Sellers.csv

In [ ]:
sellers = dataframes["sellers"]

###### Datentypen

In [ ]:
sellers.dtypes

In [ ]:
sellers.head()

In [ ]:
sellers["seller_city"].unique()

In [ ]:
sellers["seller_state"].unique()

###### Schlüssel

In [ ]:
schluessel = [
    col for col in sellers.columns
    if sellers[col].notna().all()
    and sellers[col].is_unique
]

if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für sellers ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für sellers ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(sellers)

keine Fehlenden Werte

###### Plausible Größenordnungen

In [ ]:
px.histogram(sellers, x="seller_state", color="seller_city")

Sao Paulo ist wieder nummer 1, als größte Stadt

##### Reviews.csv

In [ ]:
reviews = dataframes["reviews"]

###### Datentypen

In [ ]:
reviews.dtypes

In [ ]:
reviews.head()

In [ ]:
datumsspalten = ["review_creation_date", "review_answer_timestamp"]

for spalte in datumsspalten:
    reviews[spalte] = pd.to_datetime(reviews[spalte], format="%Y-%m-%d %H:%M:%S")

reviews.dtypes

In [ ]:
reviews["review_score"].unique()

###### Schlüssel

In [ ]:
schluessel = [
    col for col in reviews.columns
    if reviews[col].notna().all()
    and reviews[col].is_unique
]

if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für reviews ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für reviews ist die Spalte: {schluessel}")

In [ ]:
reviews["review_id"].duplicated().sum()

In [ ]:
(reviews["order_id"].value_counts() > 1).sum() 

814 review_ids kommen mehrfach vor. Dieselbe Bewertung ist also mehreren Bestellungen zugeordnet. 547 Bestellungen haben mehr als eine Bewertung, was 551 überzähligen Zeilen entspricht, weil manche Bestellungen drei Bewertungen haben. Die Tabelle hat deshalb keinen eindeutigen Schlüssel. Für die Aufgaben verwenden wir je Bestellung die letzte Bewertung.

###### Fehlende Werte

In [ ]:
msno.matrix(reviews)

Keine Fehlenden Werte

###### Plausible Größenordnungen

In [ ]:
for spalte in datumsspalten:
    print(spalte)
    print(reviews[spalte].min())
    print(reviews[spalte].max())
    print("")

In [ ]:
reviews["review_score"].unique()

Bewertungen von 1-5, über zeitraum von 2016-2018 passt

##### Products.csv

###### Datentypen

In [ ]:
products = dataframes["products"]

In [ ]:
products

In [ ]:
products.dtypes

###### Schlüssel

In [ ]:
schluessel = [
        col for col in products.columns
        if products[col].notna().all()
        and products[col].is_unique
    ]

if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für products ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für products ist die Spalte: {schluessel}")

###### Fehlende Werte

In [ ]:
msno.matrix(products)

Products_category_name kann nicht ersetzt werden, da wir die Kategorien für die neuen product_ids nicht kennen.
Product_weight_g ist ein individueller Value, weshalb er auch nicht ersetzt werden kann (kann für Berechnungen, bei denen man für alles ein Value braucht, mit dem Median ersetzt werden).
Products_photos_qty ist immer ein Null-Value, wenn products_category_name ein Null-Value ist. Warum?
Keine Ahnung. Ist auch völlig uninteressant.

In [ ]:
products.sort_values(by = "product_photos_qty", ascending =  False)

In [ ]:
products["product_category_name"].nunique()

In [ ]:
categories = dataframes["categories"]
categories.sort_values(by="product_category_name_english", ascending = False)
categories["product_category_name"].unique()

In [ ]:
products["product_category_name"] = products["product_category_name"].str.lower()
products["product_category_name"].nunique()

products["product_category_name"].sort_values().unique()

In [ ]:
products["product_category_name"] = products["product_category_name"].str.strip()
products["product_category_name"].nunique()

In [ ]:
from difflib import get_close_matches

def korrigiere_kategorien(werte, gueltige_kategorien, mindest_score=0.8):
    referenz = set(gueltige_kategorien.dropna())
    normalisiert = werte

    ersetzungen = {}
    nicht_zugeordnet = []
    for wert in normalisiert.dropna().unique():
        if wert in referenz:
            continue
        treffer = get_close_matches(wert, referenz, n=1, cutoff=mindest_score)
        if treffer:
            ersetzungen[wert] = treffer[0]
        else:
            nicht_zugeordnet.append(wert)

    return normalisiert.replace(ersetzungen), ersetzungen, nicht_zugeordnet

In [ ]:
products["kategorie_clean"], ersetzungen, rest = korrigiere_kategorien(
    products["product_category_name"],
    categories["product_category_name"], 0.8
)

print(products["kategorie_clean"].nunique())
print(ersetzungen)
print(rest)

Products.csv ist ein gutes Beispiel, da man direkt anhand der Kennzahl uniquer `product_category_name` sehen kann wie wir schrittweise unsere Berreinigung durchführen.

Wir gehen dabei von 236 auf 73, durch das enfernen von fehlerhaften Leerzeichen und anderen Schreibfehlern

#### 2.1 b Bestelltabelle (eine Zeile je Bestellung)

In [ ]:
payments = dataframes["payments"]
payments.head()

##### Item & Seller Merge

In [ ]:
positionen = items.merge(sellers, on="seller_id", how="left")

##### Positionen je Bestellung zusammenfassen

In [ ]:
items_agg = positionen.groupby("order_id").agg(
    anzahl_positionen=("order_item_id", "count"),
    anzahl_haendler=("seller_id", "nunique"),
    warenwert=("price", "sum"),
    frachtkosten=("freight_value", "sum"),
    seller_states=("seller_state", lambda s: sorted(set(s))),
).reset_index()

##### Payments je Bestellung zusammenfassen

In [ ]:
payments_agg = payments.groupby("order_id").agg(
    number_of_payments=("payment_sequential", "count"),
    payment_type=("payment_type", "nunique"),
    sum_of_rates=("payment_installments", "sum"),
    payment_value=("payment_value", "sum"),
    payment_types=("payment_type", lambda s: sorted(set(s))),
).reset_index().sort_values(by="number_of_payments", ascending=False)

##### Letzte Bewertung je Bestellung

In [ ]:
reviews_last = (
    reviews
    .sort_values(["order_id", "review_answer_timestamp", "review_id"])
    .drop_duplicates("order_id", keep="last")
)

##### Alles an orders hängen

In [ ]:
bestellungen = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(reviews_last[["order_id", "review_score", "review_answer_timestamp"]],
           on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
)

len(orders), len(bestellungen), bestellungen["order_id"].is_unique

##### Auf bestellungen mit tatsächlichem und zugesagtem Liefertermin & letzter gültiger Bewertung zischen 1-5 limitieren

In [ ]:
bestellungen.head()

In [ ]:
bestellungen = bestellungen[bestellungen["not_approved"]==False & bestellungen["review_score"].between(1,5)]



In [ ]:
bestellungen.head()

In [ ]:
len(orders), len(bestellungen), bestellungen["order_id"].is_unique

Es bleiben noch 99281 übrig

#### 2.1 c Merkmale

In [ ]:
bestellungen.head()

##### Schlechte Review (score höchstens 2)

In [ ]:
bestellungen["bad_review"] = bestellungen["review_score"] <= 2

bestellungen["bad_review"]

##### Lieferdauer

In [ ]:
bestellungen["delivery_time"] = (
    bestellungen["order_delivered_customer_date"]
    - bestellungen["order_purchase_timestamp"]
).dt.days

bins = [0, 7, 14, 21, float("inf")]

labels = ["≤ 7 Tage", "> 7 bis 14 Tage", "> 14 bis 21 Tage", "> 21 Tage"]

bestellungen["delivery_group"] = pd.cut(
    bestellungen["delivery_time"],
    bins=bins,
    labels=labels
)

bestellungen["delivery_group"].value_counts().sort_index()

##### Verspätung

In [ ]:
bestellungen["order_delay"] = (bestellungen["order_delivered_customer_date"].dt.normalize() - bestellungen["order_estimated_delivery_date"]).dt.days
bestellungen["order_delay"].describe()

In [ ]:
bestellungen.head()

##### Versand zwischen Bundesstaaten

In [ ]:
bestellungen["zwischen_staaten"] = [
    any(s != kunde for s in haendler) if isinstance(haendler, list) else pd.NA
    for kunde, haendler in zip(bestellungen["customer_state"], bestellungen["seller_states"])
]
bestellungen["zwischen_staaten"].value_counts(dropna=False)

##### Warenkorb / Fracht 

In [ ]:
bestellungen["delivery_cost_as_percentage_of_total_cost"] = bestellungen["frachtkosten"] / bestellungen["payment_value"]
bestellungen["delivery_cost_as_percentage_of_total_cost"]

## 3. Stakeholderfragen

### 3.2 Automatische Nachricht

In [ ]:
service_observation = dataframes["service_observation"]
service_observation["service_start_at"] = pd.to_datetime(service_observation["service_start_at"], format="%Y-%m-%d %H:%M:%S")
service_observation.head()

In [ ]:
orders.head()

In [ ]:
verspaetung = pd.DataFrame({
    "order_id": orders["order_id"],
    "verspaetung_tage": (
        orders["order_delivered_customer_date"].dt.normalize() - orders["order_estimated_delivery_date"]).dt.days,
})
verspaetung["verspaetet"] = verspaetung["verspaetung_tage"] > 0

verspaetung.head()

In [ ]:
order_feedback =pd.merge(service_observation, verspaetung, on="order_id", how="left").copy()
order_feedback.info

In [ ]:
order_feedback["order_id"].nunique()